<a href="https://colab.research.google.com/github/harshghugare80-lang/ML14/blob/main/Toy_Classification_Decision_Boundaries_Iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Toy Classification Problem using Decision Boundaries and Iris Dataset

Classification is a supervised Machine Learning technique in which the model learns from labelled training data and predicts a **categorical class label** for new observations.

In this practical, we first understand classification using a small **toy dataset** and manually create a decision boundary. We then use the well-known **Iris dataset** to study the effect of a **train/test split** and **k-fold cross-validation**.

The performance of the classification model is evaluated using:

- Accuracy
- Precision
- Recall
- F1-score

The practical also demonstrates how a decision boundary separates observations belonging to different classes.

## What is Classification?

Classification is a supervised learning problem in which the target variable represents one of a finite set of classes.

For example:

- Email → Spam / Not Spam
- Tumor → Benign / Malignant
- Customer → Churn / No Churn
- Flower → Setosa / Versicolor / Virginica

A classification algorithm learns a relationship between the **features (independent variables)** and the **target class (dependent variable)**.

A classifier can be represented conceptually as:

**Features → Classification Model → Predicted Class**

## Important Machine Learning Terminology

**Feature:** An input variable used by the model for prediction.

**Target / Label:** The output class that the model tries to predict.

**Training Data:** Data used to learn the model parameters.

**Testing Data:** Unseen data used to evaluate how well the trained model generalizes.

**Decision Boundary:** A boundary or surface that separates regions corresponding to different predicted classes.

**Accuracy:** Proportion of all predictions that are correct.

**Precision:** Of the observations predicted as a particular class, the proportion that actually belongs to that class.

**Recall:** Of the observations that actually belong to a class, the proportion correctly identified by the model.

**F1-score:** Harmonic mean of precision and recall.

**Cross-validation:** A resampling technique used to estimate model performance more reliably by repeatedly training and validating the model on different subsets of the data.

## Problem Statement

Implement a toy classification problem using manually defined decision boundaries. Then apply a **train/test split** and **k-fold cross-validation** to the Iris dataset.

Compare the classification performance using:

1. Accuracy
2. Precision
3. Recall
4. F1-score

Finally, visualize the decision boundary for the toy classification problem and interpret the results obtained from the Iris classification experiment.

## Part A: Toy Classification using a Manually Defined Decision Boundary

A simple two-dimensional dataset is created so that the observations belong to two classes.

For demonstration, the decision rule is:

**If x₁ + x₂ > threshold → Class 1**

**Otherwise → Class 0**

This is a simple linear decision boundary. The purpose is to understand the basic idea of how a classifier divides the feature space before using a library-based ML classifier.

In [ ]:
# Necessary Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# Creating a small toy classification dataset manually

toy_data = pd.DataFrame({
    "x1": [1, 2, 1, 2, 3, 4, 4, 5, 5, 6],
    "x2": [1, 2, 2, 3, 2, 5, 4, 5, 6, 5]
})

# Manually defining the class using a decision boundary
# Boundary rule: x1 + x2 > 6

toy_data["class"] = np.where(toy_data["x1"] + toy_data["x2"] > 6, 1, 0)

toy_data

### Understanding the Toy Dataset

The two input variables are **x1** and **x2**.

The target variable is **class**.

The decision rule is based on the linear expression:

**x1 + x2 = 6**

Therefore, the decision boundary is:

**x2 = 6 - x1**

Points on one side of the line are assigned to Class 0 and points on the other side are assigned to Class 1.

This illustrates the fundamental idea behind a decision boundary: the feature space is divided into regions, and each region corresponds to a predicted class.

In [ ]:
# Manually implement the classifier

def manual_classifier(x1, x2, threshold=6):
    if x1 + x2 > threshold:
        return 1
    else:
        return 0

toy_data["manual_prediction"] = [
    manual_classifier(x1, x2)
    for x1, x2 in zip(toy_data["x1"], toy_data["x2"])
]

toy_data

In [ ]:
# Check the manually calculated predictions

toy_accuracy = accuracy_score(toy_data["class"], toy_data["manual_prediction"])

print("Manual Classifier Accuracy:", round(toy_accuracy, 4))

In [ ]:
# Visualize the toy data and manually defined decision boundary

plt.figure(figsize=(8, 6))

for cls, label in [(0, "Class 0"), (1, "Class 1")]:
    subset = toy_data[toy_data["class"] == cls]
    plt.scatter(subset["x1"], subset["x2"], label=label, s=80)

x_values = np.linspace(0, 7, 100)
y_values = 6 - x_values

plt.plot(x_values, y_values, linewidth=2, label="Decision Boundary: x1 + x2 = 6")

plt.xlabel("Feature x1")
plt.ylabel("Feature x2")
plt.title("Toy Classification with Manual Decision Boundary")
plt.legend()
plt.grid(True)
plt.show()

## Interpretation of the Decision Boundary

The plotted line represents the boundary between the two classes.

- Points satisfying **x1 + x2 > 6** are classified as Class 1.
- Points satisfying **x1 + x2 ≤ 6** are classified as Class 0.
- The classifier does not require a complex ML algorithm for this toy example because the decision rule is explicitly defined.
- In real-world classification, the decision boundary is generally learned from training data rather than manually specified.
- The shape of the decision boundary depends on the classification algorithm and the underlying data distribution.

# Part B: Iris Dataset

The Iris dataset is a standard classification dataset containing measurements of iris flowers.

It contains four numerical features:

1. Sepal Length
2. Sepal Width
3. Petal Length
4. Petal Width

The target contains three classes:

- Setosa
- Versicolor
- Virginica

The objective is to train a classification model that predicts the species of an iris flower from its measurements.

In [ ]:
# Load the Iris dataset

iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")

print("Feature names:")
print(iris.feature_names)

print("\nTarget names:")
print(iris.target_names)

In [ ]:
# Combine features and target for inspection

data = X.copy()
data["target"] = y
data["species"] = data["target"].map(
    {0: "setosa", 1: "versicolor", 2: "virginica"}
)

data.head()

### What are the Features?

- **Sepal length:** Length of the sepal in centimetres.
- **Sepal width:** Width of the sepal in centimetres.
- **Petal length:** Length of the petal in centimetres.
- **Petal width:** Width of the petal in centimetres.

### What is the Response / Target?

The target is the flower species:

- 0 → Setosa
- 1 → Versicolor
- 2 → Virginica

The Iris problem is therefore a **multi-class classification problem**.

In [ ]:
# Check the shape of the dataset

print("Dataset shape:", data.shape)

# Check data types and non-null values
data.info()

In [ ]:
# Check for missing values

data.isna().sum()

In [ ]:
# Check class distribution

data["species"].value_counts()

## Exploratory Visualization

Before training a classifier, it is useful to inspect whether the classes appear separable in feature space.

For a two-dimensional decision-boundary visualization, petal length and petal width are selected because they provide a useful representation of the class separation.

In [ ]:
# Plot Iris classes using petal length and petal width

plt.figure(figsize=(8, 6))

for cls, label in enumerate(iris.target_names):
    subset = data[data["target"] == cls]
    plt.scatter(
        subset["petal length (cm)"],
        subset["petal width (cm)"],
        label=label,
        s=60
    )

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Iris Dataset: Petal Length vs Petal Width")
plt.legend()
plt.grid(True)
plt.show()

# Part C: Train/Test Split

A train/test split divides the dataset into two parts.

**Training set:** Used to train the classification model.

**Testing set:** Used only after training to estimate performance on unseen observations.

A common choice is 80% training data and 20% testing data.

The `stratify=y` parameter is used so that the class proportions are approximately preserved in both subsets.

In [ ]:
# Split the Iris dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

## Logistic Regression Classifier

Logistic Regression is a supervised classification algorithm.

Although its name contains the word regression, it is commonly used for classification because it estimates class probabilities and assigns observations to classes.

For a multi-class problem such as Iris, Logistic Regression can use a multinomial strategy to model the probability of each class.

The model learns the relationship between the four flower measurements and the flower species.

In [ ]:
# Create and train the Logistic Regression model

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model training completed.")

In [ ]:
# Make predictions on the test data

y_pred = model.predict(X_test)

print("First 10 actual labels    :", y_test.to_numpy()[:10])
print("First 10 predicted labels :", y_pred[:10])

# Model Evaluation on Test Data

For a multi-class classification problem, we can calculate the following measures:

### Accuracy

Accuracy = Correct Predictions / Total Predictions

### Precision

Precision measures how many of the observations predicted as a class are actually members of that class.

### Recall

Recall measures how many observations belonging to a class are correctly identified.

### F1-score

F1-score is the harmonic mean of precision and recall.

For the multi-class Iris problem, **macro averaging** gives equal importance to each class.

In [ ]:
# Calculate evaluation metrics

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, average="macro")
test_recall = recall_score(y_test, y_pred, average="macro")
test_f1 = f1_score(y_test, y_pred, average="macro")

print("Test Set Performance")
print("--------------------")
print("Accuracy :", round(test_accuracy, 4))
print("Precision:", round(test_precision, 4))
print("Recall   :", round(test_recall, 4))
print("F1-score :", round(test_f1, 4))

In [ ]:
# Detailed classification report

print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names
    )
)

In [ ]:
# Confusion matrix

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
)

disp.plot()
plt.title("Confusion Matrix - Iris Test Set")
plt.show()

## K-Fold Cross-Validation

A single train/test split can produce a performance estimate that depends on which observations happen to be selected for testing.

**K-fold cross-validation** reduces this dependence.

In k-fold cross-validation:

1. The dataset is divided into k approximately equal folds.
2. One fold is used for validation.
3. The remaining k−1 folds are used for training.
4. The process is repeated k times.
5. Each observation is used for validation once.
6. The scores from all folds are summarized using the mean and standard deviation.

For classification, **StratifiedKFold** is preferred because it attempts to preserve the class distribution in each fold.

Here, **5-fold cross-validation** is used.

In [ ]:
# Define 5-fold stratified cross-validation

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision_macro",
    "recall": "recall_macro",
    "f1": "f1_macro"
}

cv_results = cross_validate(
    model,
    X,
    y,
    cv=cv,
    scoring=scoring
)

print("5-Fold Cross-Validation Results")
print("--------------------------------")

for metric in scoring:
    scores = cv_results["test_" + metric]
    print(
        metric.capitalize(),
        ": Mean =",
        round(scores.mean(), 4),
        ", Std =",
        round(scores.std(), 4)
    )

## Compare Train/Test Split and K-Fold Cross-Validation

The train/test split provides one estimate of model performance on unseen data.

K-fold cross-validation provides several validation estimates and summarizes them using the mean and standard deviation.

A comparison can therefore be made using the four requested metrics:

- Accuracy
- Precision
- Recall
- F1-score

The following table combines the test-set results and the mean cross-validation results.

In [ ]:
# Create a comparison table

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Train/Test Split": [
        test_accuracy,
        test_precision,
        test_recall,
        test_f1
    ],
    "5-Fold CV Mean": [
        cv_results["test_accuracy"].mean(),
        cv_results["test_precision"].mean(),
        cv_results["test_recall"].mean(),
        cv_results["test_f1"].mean()
    ],
    "5-Fold CV Std": [
        cv_results["test_accuracy"].std(),
        cv_results["test_precision"].std(),
        cv_results["test_recall"].std(),
        cv_results["test_f1"].std()
    ]
})

comparison.round(4)

In [ ]:
# Visual comparison of test-set and cross-validation mean scores

plot_data = comparison.set_index("Metric")[[
    "Train/Test Split",
    "5-Fold CV Mean"
]]

plot_data.plot(
    kind="bar",
    figsize=(9, 6)
)

plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Train/Test Split vs 5-Fold Cross-Validation")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

# Decision Boundary Visualization for Iris

The Iris dataset has four features, so its complete decision boundary exists in a four-dimensional feature space.

For visualization, we use only **petal length** and **petal width**. A new Logistic Regression model is trained using these two features.

The resulting two-dimensional plot shows the predicted class region and the observations from the Iris dataset.

In [ ]:
# Select two features for a 2-D decision-boundary visualization

X_2d = data[
    ["petal length (cm)", "petal width (cm)"]
]
y_2d = data["target"]

boundary_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

boundary_model.fit(X_2d, y_2d)

print("2-D decision-boundary model trained.")

In [ ]:
# Create a mesh grid and predict each point in the feature space

x_min, x_max = X_2d.iloc[:, 0].min() - 0.5, X_2d.iloc[:, 0].max() + 0.5
y_min, y_max = X_2d.iloc[:, 1].min() - 0.5, X_2d.iloc[:, 1].max() + 0.5

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400)
)

grid = np.c_[xx.ravel(), yy.ravel()]
Z = boundary_model.predict(grid)
Z = Z.reshape(xx.shape)

In [ ]:
# Plot the learned decision regions

plt.figure(figsize=(9, 6))

plt.contourf(
    xx,
    yy,
    Z,
    alpha=0.25
)

for cls, label in enumerate(iris.target_names):
    subset = data[data["target"] == cls]
    plt.scatter(
        subset["petal length (cm)"],
        subset["petal width (cm)"],
        label=label,
        s=60
    )

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Decision Boundaries for Iris Classification")
plt.legend()
plt.grid(True)
plt.show()

# Result and Discussion

The practical demonstrates two important ideas in classification.

### 1. Manual Decision Boundary

The toy example shows that a simple classification rule can divide the feature space into separate class regions. This provides an intuitive understanding of a **decision boundary**.

### 2. Iris Classification

The Iris dataset is classified using Logistic Regression. The model is first evaluated using a train/test split and then using 5-fold stratified cross-validation.

The four evaluation measures—**accuracy, precision, recall and F1-score**—provide complementary information about classifier performance.

### 3. Train/Test Split vs Cross-Validation

The train/test score represents performance on one particular held-out subset. The cross-validation mean provides a more stable estimate based on multiple train/validation partitions.

The cross-validation standard deviation indicates how much the model's performance changes across folds. A smaller standard deviation generally indicates more consistent performance.

### 4. Decision Boundary

The two-dimensional Iris visualization demonstrates how a classification model divides feature space into predicted class regions. In the complete Iris model, the classifier uses all four features, so the actual decision surface cannot be represented completely in a two-dimensional graph.

## Conclusion

Thus, a toy classification problem was implemented using a manually defined decision boundary, followed by Iris classification using a train/test split and 5-fold cross-validation. Accuracy, precision, recall and F1-score were calculated and compared to understand classifier performance and the importance of robust model evaluation.

# Viva / Practical Questions

1. What is a classification problem?
2. What is a decision boundary?
3. Why is the toy classifier called a manually defined classifier?
4. What is the difference between training data and testing data?
5. Why is `stratify=y` used in the train/test split?
6. What is k-fold cross-validation?
7. Why is StratifiedKFold suitable for classification?
8. What is the difference between accuracy and precision?
9. What is the difference between precision and recall?
10. Why is F1-score useful?
11. What does macro averaging mean in a multi-class problem?
12. Why can the test-set score differ from the cross-validation mean?
13. What does the standard deviation of cross-validation scores indicate?
14. Why are only two Iris features used for the decision-boundary plot?
15. Why can the complete four-feature Iris decision boundary not be directly visualized in two dimensions?